In [1]:
import re
from util_order import *
import pandas as pd
import random
from cal_sel import *
import os
import logging
from llm import *

In [ ]:
sql_query = "YOUR SQL QUERY HERE"
result_dir = "YOUR RESULT DIRECTORY HERE"
##############################################
# use main.py to generate the candidate files#
##############################################
candidate_dir_A = "Your candidate directory A"
candidate_dir_B = "Your candidate directory B"
candidate_dir_C = "Your candidate directory C"
candidate_dir_D = "Your candidate directory D"
#....more candidate directories if needed
OPENAI_KEY = 'your_api_key'
init_chatgpt(OPENAI_KEY)

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
if not os.path.exists(candidate_dir_A):
    os.makedirs(candidate_dir_A)
if not os.path.exists(candidate_dir_B):
    os.makedirs(candidate_dir_B)
if not os.path.exists(candidate_dir_C):
    os.makedirs(candidate_dir_C)
if not os.path.exists(candidate_dir_D):
    os.makedirs(candidate_dir_D)
#....more candidate directories if needed
dataA = pd.read_csv(candidate_dir_A)
dataB = pd.read_csv(candidate_dir_B)
dataC = pd.read_csv(candidate_dir_C)
dataD = pd.read_csv(candidate_dir_D)


select_clause = re.findall(r"SELECT(.*?)FROM", sql_query, re.DOTALL)[0].strip()
from_match = re.search(r"FROM(.*?)(WHERE|JOIN|$)", sql_query, re.DOTALL)
from_clause = from_match.group(1).strip() if from_match else ''
where_match = re.search(r"WHERE(.*)", sql_query, re.DOTALL)
where_clause = where_match.group(1).strip() if where_match else ''
join_on_matches = re.findall(r"(JOIN .*? ON .*?)(?=(JOIN|WHERE|$))", sql_query, re.DOTALL)
join_clauses = [match[0].strip() for match in join_on_matches]

In [3]:
from dataclasses import dataclass
from typing import List

@dataclass
class JoinInfor:
    jointable1: str
    jointable2: str
    join_clause: str
    join_table1_attributes: str
    join_table2_attributes: str
    selectivity_join: float
    E_value: float

@dataclass
class TableInfo:
    name: str
    file_list: list
    file_path: str
    file_data_path: str
    data: dict
    where_filter_conditions: list
    where_clause: str
    select_attributes: list
    join_infor: List[JoinInfor]
    

In [4]:
############################################
#Emin = cal_Emin(ordered_filters)
############################################
def cal_Emin(ordered_filters):
    Emin = 0
    sel = 1
    for first_table_filter_data in ordered_filters:
        Emin += sel * first_table_filter_data["cost"]
        sel *= (1 - first_table_filter_data["selectivity"])
    return Emin

############################################
#selectivity_join_A = cal_selectivity_join(A_join_attributes,B_join_attributs, dataA, dataB)
############################################
def cal_selectivity_join(A_join_attributes, B_join_attributes, dataA, dataB, is_table1_first):
    dataA[A_join_attributes] = dataA[A_join_attributes].astype(str)
    dataB[B_join_attributes] = dataB[B_join_attributes].astype(str)
    if is_table1_first:
        join_data = pd.merge(dataA, dataB, how='inner', left_on=A_join_attributes, right_on=B_join_attributes)
        selectivity_join = len(join_data) / len(dataA) 
    else:
        join_data = pd.merge(dataA, dataB, how='inner', left_on=B_join_attributes, right_on=A_join_attributes)
        selectivity_join = len(join_data) / len(dataB)  
    
    return selectivity_join

In [5]:
#########################################################
##### you need change the function to fit your data #####
#########################################################
def get_table_num(table_name):
    if table_name == "A":
        return 0
    elif table_name == "B":
        return 1
    elif table_name == "C":
        return 2
    elif table_name == "D":
        return 3
    else:
        return -1

In [6]:
#########################################################
##### you need change the function to fit your data #####
#########################################################
def parse_join_clause(join_clause, dataA=None, dataB=None):
    join_condition_match = re.search(r"JOIN (\w+) ON (\w+)\.(\w+) = (\w+)\.(\w+)", join_clause)
    if join_condition_match:
        jointable2 = join_condition_match.group(1)
        jointable1 = join_condition_match.group(2)
        join_table1_attributes = join_condition_match.group(3)
        jointable2_alias = join_condition_match.group(4)
        join_table2_attributes = join_condition_match.group(5)
        
        join_clause_text = f"{jointable1}.{join_table1_attributes} = {jointable2_alias}.{join_table2_attributes}"

        return JoinInfor(
            jointable1=jointable1,
            jointable2=jointable2_alias,
            join_clause=join_clause_text,
            join_table1_attributes=join_table1_attributes,
            join_table2_attributes=join_table2_attributes,
            selectivity_join=0,
            E_value=0
        )

In [7]:
############################################
#Eall = cal_Eall(file_list_A, file_list_B,file_path_A, file_path_B, dataA,dataB, selectivity_join_A, selectivity_join_B)
############################################
import os
import time

def cal_Eall(table_info_A, table_info_B,join_selectivity_A, join_selectivity_B,join_clause):
    join_infor = parse_join_clause(join_clause)

    file_list_A = table_info_A.file_list
    file_path_A = table_info_A.file_path
    dataA = table_info_A.data
    selectivity_join_A = join_selectivity_A
    A_where_filter_conditions = table_info_A.where_filter_conditions
    where_A_clause = table_info_A.where_clause
    A_join_attributes = join_infor.join_table1_attributes

    file_list_B = table_info_B.file_list
    file_path_B = table_info_B.file_path
    dataB = table_info_B.data
    selectivity_join_B = join_selectivity_B
    B_where_filter_conditions = table_info_B.where_filter_conditions
    where_B_clause = table_info_B.where_clause
    B_join_attributes = join_infor.join_table2_attributes

    first_table_filter_data = {}
    start_time = time.time()
    
    Emin = 0
    cost_join_attribute = 0
    E_all = 0

    for onefile in file_list_A:
        input_text = ""
        file_path = os.path.join(file_path_A, onefile)
        with open(file_path, "r") as file:
            input_text = file.read()

        cur_file_attributes = parse_input(input_text)
        for filter_condition in A_where_filter_conditions:
            table, cur_filter_condition = filter_condition.split(".")
            attribute, operator, value = cur_filter_condition.split(maxsplit=2)
            selectivity = cal_sel(dataA, {"name": cur_filter_condition})
            if attribute in cur_file_attributes:
                try:
                    cur_sentence = cur_file_attributes[attribute]['key_sentences']
                    cur_cost = cur_file_attributes[attribute]['cost']
                except KeyError:
                    cur_sentence = ""
                    cur_cost = 0
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": cur_cost,
                    's_value': calculate_s(selectivity, cur_sentence),
                    "key_sentences": cur_sentence
                }
            else:
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": 0,
                    's_value': 0,
                    "key_sentences": []
                }
        try:
            sorted_filters = handle_sql(where_A_clause, first_table_filter_data)
        except Exception as e:
            continue

        
        ordered_filters = []
        for filter_cond, s_value in sorted_filters:
            order_filter = first_table_filter_data[filter_cond]
            ordered_filters.append(order_filter)
    
        Emin += cal_Emin(ordered_filters)
    
        try:
            cur_cost_join_attribute = cur_file_attributes[A_join_attributes]['cost']
        except KeyError:
            cur_cost_join_attribute = 0
        cost_join_attribute += cur_cost_join_attribute
    
    first_table_filter_data = {}

    for onefile in file_list_B:
        input_text = ""
        file_path = os.path.join(file_path_B, onefile)
        #print("file:", file_path)
        with open(file_path, "r") as file:
            input_text = file.read()
        
        cur_file_attributes = parse_input(input_text)
    
        #print("filter conditions:", B_where_filter_conditions)
        for filter_condition in B_where_filter_conditions:
            table, cur_filter_condition = filter_condition.split(".")
            attribute, operator, value = cur_filter_condition.split(maxsplit=2)
            #print(f"attribute: {attribute}, operator: {operator}, value: {value}")
            selectivity = cal_sel(dataB, {"name": cur_filter_condition})
            if attribute in cur_file_attributes:
                try:
                    cur_sentence = cur_file_attributes[attribute]['key_sentences']
                    cur_cost = cur_file_attributes[attribute]['cost']
                except KeyError:
                    cur_sentence = ""
                    cur_cost = 0
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": cur_cost,
                    's_value': calculate_s(selectivity, cur_sentence),
                    "key_sentences": cur_sentence
                }
            else:
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": 0,
                    's_value': 0,
                    "key_sentences": []
                }
        if B_join_attributes in cur_file_attributes:
            try:
                cur_cost_join_attribute = cur_file_attributes[B_join_attributes]['cost']
                cur_sentence = cur_file_attributes[B_join_attributes]['key_sentences']
            except KeyError:
                cur_cost_join_attribute = 0
                cur_sentence = ""
            first_table_filter_data["join_filter_condition"] = {
                "name": "join_filter_condition",
                "table": "B",
                "selectivity": selectivity_join_B,
                "cost": cur_cost_join_attribute,
                's_value': calculate_s(selectivity_join_B, cur_sentence),
                "key_sentences": cur_sentence
            }
    
        #print("first_table_filter_data:", first_table_filter_data)
        try:
            #print("where_B_clause:", where_B_clause)
            sorted_filters = handle_sql(where_B_clause, first_table_filter_data)
        except Exception as e:
            print("Error processing file:", file_path, "Error:", e)
            continue
        # print("Sorted Filters:")
        # for filter_cond, s_value in sorted_filters:
        #     print(f"{filter_cond}: S = {s_value:.4f}")
        
        ordered_filters = []
        for filter_cond, s_value in sorted_filters:
            order_filter = first_table_filter_data[filter_cond]
            ordered_filters.append(order_filter)
        #print("ordered_filters:", ordered_filters)
    
        Emin += cal_Emin(ordered_filters)
       
    E_all = Emin + cost_join_attribute*selectivity_join_A
    #print("E_all:", E_all)
    return E_all

In [8]:
##############################################
#A_select_attributes, B_select_attributes = get_select_attributes(select_clause)
##############################################
def get_select_attributes(select_clause, from_clause):
    sql_attributes = select_clause.split(",")
    table_names = from_clause.split(",")
    table_select_attributes = {table.strip(): [] for table in table_names}

    for attribute in sql_attributes:
        attribute = attribute.strip()
        if "." in attribute:
            table_alias, column = attribute.split(".")
            table_alias = table_alias.strip()
            column = column.strip()
            if table_alias in table_select_attributes:
                table_select_attributes[table_alias].append(column)

    return table_select_attributes

def get_filter_conditions(where_clause, from_clause):
    conditions = where_clause.split("AND")
    table_names = from_clause.split(",")
    
    table_filter_conditions = {table.strip(): [] for table in table_names}

    for condition in conditions:
        condition = condition.strip()
        if "." in condition:
            table_alias = condition.split(".")[0].strip()
            if table_alias in table_filter_conditions:
                table_filter_conditions[table_alias].append(condition)

    return table_filter_conditions

def InitiTableInfo(from_clause, select_clause, where_clause, file_data_paths,file_paths, join_clauses,file_list_dir):
    table_names = [table.strip() for table in from_clause.split(",")]
    table_infos = []
    
    select_attributes = get_select_attributes(select_clause, from_clause)
    filter_conditions = get_filter_conditions(where_clause, from_clause)

    table_data = {}
    for i, table_name in enumerate(table_names):
        table_data[table_name] = pd.read_csv(file_data_paths[i], encoding='utf-8')
        table_info = TableInfo(
            name=table_name,
            file_list=os.listdir(file_list_dir[i]),
            file_path=file_paths[i],
            file_data_path=file_data_paths[i],
            data=table_data[table_name],
            where_filter_conditions=filter_conditions.get(table_name, []),
            where_clause=" AND ".join(filter_conditions.get(table_name, [])),
            select_attributes=select_attributes.get(table_name, []),
            join_infor=[]
        )
        table_infos.append(table_info)

    for join_clause in join_clauses:
        join_info = parse_join_clause(join_clause)
        for table_info in table_infos:
            current_join_info = JoinInfor(
                jointable1=join_info.jointable1,
                jointable2=join_info.jointable2,
                join_clause=join_info.join_clause,
                join_table1_attributes=join_info.join_table1_attributes,
                join_table2_attributes=join_info.join_table2_attributes,
                selectivity_join=0, 
                E_value=0
            )
            table_1_num = get_table_num(current_join_info.jointable1)
            table_2_num = get_table_num(current_join_info.jointable2)
          
            table_1_data = pd.read_csv(file_data_paths[table_1_num], encoding='utf-8')
            table_2_data = pd.read_csv(file_data_paths[table_2_num], encoding='utf-8')
            table1_data_cur = table_1_data
            #print(f"table1_data: {current_join_info.join_table1_attributes}")
            table2_data_cur = table_2_data
            #print(f"table2_data: {current_join_info.join_table2_attributes}")
            selectivity_A = cal_selectivity_join(
                current_join_info.join_table1_attributes,
                current_join_info.join_table2_attributes,
                table1_data_cur, table2_data_cur,
                is_table1_first=True  # 
            )
            selectivity_B = cal_selectivity_join(
                current_join_info.join_table1_attributes,
                current_join_info.join_table2_attributes,
                table1_data_cur, table2_data_cur,
                is_table1_first=False  
            )
            
            if table_info.name == current_join_info.jointable1:
                current_join_info.selectivity_join = selectivity_A
                #print(f"table: {table_info.name}, selectivity_join: {selectivity_A}")
               
                E_value = cal_Eall(table_info, [t for t in table_infos if t.name == current_join_info.jointable2][0],selectivity_A, selectivity_B,join_clause)
                current_join_info.E_value = E_value
                #print(f"table: {table_info.name}, E_value: {E_value}")

                table_info.join_infor.append(current_join_info)

            elif table_info.name == current_join_info.jointable2:
                current_join_info.selectivity_join = selectivity_B
                #print(f"table: {table_info.name}, selectivity_join: {selectivity_B}")
               
                E_value = cal_Eall(table_info, [t for t in table_infos if t.name == current_join_info.jointable2][0],selectivity_A, selectivity_B,join_clause)
                current_join_info.E_value = E_value
                #print(f"table: {table_info.name}, E_value: {E_value}")

                table_info.join_infor.append(current_join_info)

    return table_infos


In [ ]:
################################################################
##### you need to change the file path to fit your data #######
################################################################
file_data_pathss = ["your file path A", "your file path B", "your file path C", "your file path D"]
file_pathss = ["your file path A", "your file path B", "your file path C", "your file path D"]
file_list_dir = ["your file list A", "your file list B", "your file list C", "your file list D"]
table_infos = InitiTableInfo(from_clause, select_clause, where_clause, file_data_pathss, file_pathss, join_clauses,file_list_dir)

for table_info in table_infos:
    print(f"Table {table_info.name}:")
    #print("file path:", table_info.file_data_path)
    print(f"  Select Attributes: {table_info.select_attributes}")
    print(f"  Where Conditions: {table_info.where_filter_conditions}")
    print(f"  Join Info: {table_info.join_infor}")
    print()

In [ ]:

#first_table_info,second_table_info = get_2_table_order(table_info_A, table_info_B)
first_table_info, second_table_info = table_infos[0], table_infos[1]
first_table_path = first_table_info.file_path
second_table_path = second_table_info.file_path

print("first_table_infor:", first_table_path)
print("second_table_infor:", second_table_path)


In [11]:
def rewrite_where_clause(where_clause):
    tmp_filter_cond = extract_filter_condition(where_clause)
    filter_conds = []
    for filter_cond in tmp_filter_cond:
        table, sub_filter_condition = filter_cond.split(".", 1)
        filter_conds.append(sub_filter_condition)
    where_clause = "WHERE " + " AND ".join(filter_conds)
    return where_clause

def rewrite_where_filter_conditions(where_filter_conditions):
    filter_conds = []
    for filter_cond in where_filter_conditions:
        table, sub_filter_condition = filter_cond.split(".", 1)
        filter_conds.append(sub_filter_condition)
    return filter_conds

In [ ]:
first_table_filter_data = {}
all_total_token = 0
all_actual_token = 0
if not os.path.exists(result_dir):
    os.makedirs(result_dir)
first_table_output = pd.DataFrame()

import os
import time
file_list = os.listdir(first_table_path)

infor = result_dir + "infor.txt"
start_tie = time.time()
initial_where_clause = where_clause

for onefile in file_list:
    input_text = ""
    file_path = first_table_path + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    where_clause = first_table_info.where_clause
    where_clause = rewrite_where_clause(where_clause)
    print("where_clause: ",where_clause)

    filter_conditions = first_table_info.where_filter_conditions
    filter_cond = rewrite_where_filter_conditions(filter_conditions)
    print("filter_cond: ",filter_cond)

    data = first_table_info.data


    attributes = parse_input(input_text)

    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    for filter_condtion in filter_cond:
        print("filter_condtion: ",filter_condtion)
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        try:
            key_sentences = attributes[atrribute]['key_sentences']
            cost = attributes[atrribute]['cost']
        except KeyError:
            key_sentences = ""
            cost = 0
        if atrribute in attributes:
            first_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "table": first_table_info.name,
                "selectivity": selectivity,
                "cost": cost,
                's_value': calculate_s(selectivity, key_sentences),
                "key_sentences": key_sentences
            }
        else:
            first_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }
    #print("first_table_filter_data: ",first_table_filter_data)

    print("first_table_filter_data: ",first_table_filter_data)
    print("where_clause: ",where_clause)
    try:
        sort_where_clause = where_clause.replace("WHERE","")
        sorted_filters = handle_sql(sort_where_clause, first_table_filter_data)
    except:
        print("the file is error: ",file_path)
        continue
    print("Sorted Filters:")
    for filter_cond, s_value in sorted_filters:
        print(f"{filter_cond}: S = {s_value:.4f}")
    #continue
    #break
    print("\n")

    actual_token = 0

    skipthefile = 0
    sql_copy = where_clause

    select_attributes = first_table_info.select_attributes
    remaining_attributes = select_attributes.copy()

    remaining_attributes = list(set(remaining_attributes))
    first_table_curdata = {}
    first_table_sorted_filter_cond = []
    first_table_mapfilter_cond = {}
    first_table_mapfilter_erro = {}

    for filter in sorted_filters:
        first_table_sorted_filter_cond.append(filter[0])
    for filter in first_table_sorted_filter_cond:
        first_table_mapfilter_cond[filter] = 0
        first_table_mapfilter_erro[filter] = 0

    remaining_first_table_sorted_filter_cond = first_table_sorted_filter_cond.copy()

    while remaining_first_table_sorted_filter_cond:
        #print("remaining_first_table_sorted_filter_cond: ",remaining_first_table_sorted_filter_cond)
        filter_cond_tochange = []
        filter_cond = remaining_first_table_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        first_table_mapfilter_cond[filter_cond] += 1
        if first_table_mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remaining_first_table_sorted_filter_cond:
                remaining_first_table_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif first_table_mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remaining_first_table_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            print("answer: \n", answer)
            #attributes_cur_all = answer.split("$$")[1]
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                # if 'first_table_output NOT IN REQUIRED FORMAT' in answer and first_table_mapfilter_erro[filter_cond] >= 1:
                #     if filter_cond in remaining_first_table_sorted_filter_cond:
                #         remaining_first_table_sorted_filter_cond.remove(filter_cond)
                first_table_mapfilter_cond[filter_cond] -= 1
                first_table_mapfilter_erro[filter_cond] += 1
                tochange_filter = [filter_cond,'false']
                actual_token -= len(key_sentences.split())+100
                # filter_cond_tochange.append(tochange_filter)
                
                # for ask_filter_cond,bool_value in filter_cond_tochange:
                #         sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
                sql_copy = sql_copy.replace(filter_cond,'false')
                sql_copy2 = sql_copy
                print("sql_copy: ",sql_copy)
                bool_value_set_true = calculate_bool_value_true(sql_copy)
                bool_value_set_false = calculate_bool_value_false(sql_copy2)
                print("bool_value_set_true: ",bool_value_set_true)
                print("bool_value_set_false: ",bool_value_set_false)

                if bool_value_set_true == False:
                    skipthefile = 1
                    break

                if bool_value_set_false == True:
                    skipthefile = 0
                    break
                else:
                    skipthefile = 1
                continue

            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    first_table_curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            if dataattr == first_table_info.join_attributes:
                                remaining_attributes.remove(dataattr)
                            #remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remaining_first_table_sorted_filter_cond:
                            #remaining_first_table_sorted_filter_cond.remove(filter_cond_cur)
                            if filter_cond_cur == filter_cond:
                                filter_cond_tochange.append(tochange_filter)
                        print("remaining_first_table_sorted_filter_cond: ",remaining_first_table_sorted_filter_cond)
                    else:
                        filter_cond_tochange.append([filter_cond,'false'])
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)

        remaining_first_table_sorted_filter_cond.remove(filter_cond)
        if bool_value_set_true == False:
            skipthefile = 1
            break
        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")

    if skipthefile == 0:
        
        
        key_sentences = ""
        first_table_mapattr = {}
        for attr in select_attributes:
            first_table_mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            attr = remaining_attributes[0]
            first_table_mapattr[attr] += 1
            if first_table_mapattr[attr] >= 2:
                remaining_attributes.remove(attr)
                first_table_curdata[attr] = "NAN"
                continue
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+75
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        first_table_curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                                #remaining_attributes.remove(dataattr)
                            else:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                except:
                    print("error extract attributes")
                    actual_token -= len(key_sentences.split())+75
                    print("answer: ", answer) 
       
             
        if len(first_table_curdata) != 0:
            first_table_curdata['file'] = onefile
            first_table_output = first_table_output.append(first_table_curdata,ignore_index=True)
            data = data.append(first_table_curdata,ignore_index=True)
            first_table_output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)

with open(infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

first_table_output


In [ ]:
join_attr = first_table_info.join_infor[0].join_table1_attributes
print("join_attr: ",join_attr)
join_attributes = first_table_output[join_attr].values
join_attributes = list(set(join_attributes))
print("join_attributes: ",join_attributes)

join_attributes = [str(i) for i in join_attributes]
join_attributes = ",".join(join_attributes)
join_filter_conditions = join_attr + " IN " + str(join_attributes)
print("join_attributes: ",join_attributes)
print("join_filter_conditions: ",join_filter_conditions)

join_attr_next = second_table_info.join_infor[1].join_table1_attributes
print("join_attr_next: ",join_attr_next)
join_attr2_next = second_table_info.join_infor[0].join_table1_attributes
print("join_attr2_next: ",join_attr2_next)
join_attr3_next = second_table_info.join_infor[2].join_table1_attributes
print("join_attr3_next: ",join_attr3_next)


In [ ]:
second_table_filter_data = {}
second_table_output = pd.DataFrame()
all_actual_token = 0
import os
import time
second_file_list = os.listdir(second_table_path)

second_infor = result_dir + "second_infor.txt"
start_tie = time.time()
initial_where_clause = where_clause

for onefile in second_file_list:
    input_text = ""
    file_path = second_table_path + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    where_clause = second_table_info.where_clause
    where_clause = rewrite_where_clause(where_clause)
    print("where_clause: ",where_clause)

    filter_conditions = second_table_info.where_filter_conditions
    filter_cond = rewrite_where_filter_conditions(filter_conditions)
    print("filter_cond: ",filter_cond)

    data = second_table_info.data

    attributes = parse_input(input_text)

    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    for filter_condtion in filter_cond:
        print("filter_condtion: ",filter_condtion)
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        try:
            key_sentences = attributes[atrribute]['key_sentences']
            cost = attributes[atrribute]['cost']
        except KeyError:
            key_sentences = ""
            cost = 0
        if atrribute in attributes:
            second_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "table": second_table_info.name,
                "selectivity": selectivity,
                "cost": cost,
                's_value': calculate_s(selectivity, key_sentences),
                "key_sentences": key_sentences
            }
        else:
            second_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }
    #print("second_table_filter_data: ",second_table_filter_data)
    
    try:
            key_sentences = attributes[join_attr]['key_sentences']
            cost = attributes[join_attr]['cost']
    except KeyError:
        key_sentences = ""
        cost = 0
    if atrribute in attributes:
        second_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "table": second_table_info.name,
            "selectivity": second_table_info.join_infor[2].selectivity_join,
            "cost": cost,
            's_value': calculate_s(selectivity, key_sentences),
            "key_sentences": key_sentences
        }
    else:
        second_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "selectivity": second_table_info.selectivity_join,
            's_value': 0,
            "key_sentences": []
        }


    print("second_table_filter_data: ",second_table_filter_data)
    print("where_clause: ",where_clause)
    try:
        where_clause += " AND " + join_filter_conditions
        sort_where_clause = where_clause.replace("WHERE","")
        print("sort_where_clause: ",sort_where_clause)
        sorted_filters = handle_sql(sort_where_clause, second_table_filter_data)
    except:
        print("the file is error: ",file_path)
        continue
    print("Sorted Filters:")
    for filter_cond, s_value in sorted_filters:
        print(f"{filter_cond}: S = {s_value:.4f}")
    #continue
    #break
    print("\n")

    actual_token = 0
    
    

    skipthefile = 0
    sql_copy = where_clause
    
    select_attributes = second_table_info.select_attributes
    remaining_attributes = select_attributes.copy()
    remaining_attributes.append(join_attr)
    remaining_attributes.append(join_attr_next)
    remaining_attributes.append(join_attr2_next)
    remaining_attributes.append(join_attr3_next)
    print("remaining_attributes: ",remaining_attributes)
    
    remaining_attributes = list(set(remaining_attributes))
    second_table_curdata = {}
    second_table_sorted_filter_cond = []
    second_table_mapfilter_cond = {}
    second_table_mapfilter_erro = {}

    for filter in sorted_filters:
        second_table_sorted_filter_cond.append(filter[0])
    for filter in second_table_sorted_filter_cond:
        second_table_mapfilter_cond[filter] = 0
        second_table_mapfilter_erro[filter] = 0
    
    remaining_second_table_sorted_filter_cond = second_table_sorted_filter_cond.copy()

    while remaining_second_table_sorted_filter_cond:
        #print("remaining_second_table_sorted_filter_cond: ",remaining_second_table_sorted_filter_cond)
        filter_cond_tochange = []
        filter_cond = remaining_second_table_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        second_table_mapfilter_cond[filter_cond] += 1
        if second_table_mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remaining_second_table_sorted_filter_cond:
                remaining_second_table_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif second_table_mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remaining_second_table_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            print("answer: \n", answer)
            #attributes_cur_all = answer.split("$$")[1]
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                # if 'second_table_output NOT IN REQUIRED FORMAT' in answer and second_table_mapfilter_erro[filter_cond] >= 1:
                #     if filter_cond in remaining_second_table_sorted_filter_cond:
                #         remaining_second_table_sorted_filter_cond.remove(filter_cond)
                second_table_mapfilter_cond[filter_cond] -= 1
                second_table_mapfilter_erro[filter_cond] += 1
                tochange_filter = [filter_cond,'false']
                actual_token -= len(key_sentences.split())+100
                # filter_cond_tochange.append(tochange_filter)
                
                # for ask_filter_cond,bool_value in filter_cond_tochange:
                #         sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
                sql_copy = sql_copy.replace(filter_cond,'false')
                sql_copy2 = sql_copy
                print("sql_copy: ",sql_copy)
                bool_value_set_true = calculate_bool_value_true(sql_copy)
                bool_value_set_false = calculate_bool_value_false(sql_copy2)
                print("bool_value_set_true: ",bool_value_set_true)
                print("bool_value_set_false: ",bool_value_set_false)
                
                if bool_value_set_true == False:
                    skipthefile = 1
                    break
                
                if bool_value_set_false == True:
                    skipthefile = 0
                    break
                else:
                    skipthefile = 1
                continue

            
            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    second_table_curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            if dataattr == second_table_info.join_attributes:
                                remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remaining_second_table_sorted_filter_cond:
                            if filter_cond_cur == filter_cond:
                                #remaining_second_table_sorted_filter_cond.remove(filter_cond)
                                filter_cond_tochange.append(tochange_filter)
                        print("remaining_second_table_sorted_filter_cond: ",remaining_second_table_sorted_filter_cond)
                    else:
                        filter_cond_tochange.append([filter_cond,'false'])
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                filter_cond_tochange.append([filter_cond,'false'])
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)

        remaining_second_table_sorted_filter_cond.remove(filter_cond)
        if bool_value_set_true == False:
            skipthefile = 1
            break
        
        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")
        #remaining_second_table_sorted_filter_cond.remove(filter_cond)



    
    if skipthefile == 0:
        
        
        key_sentences = ""
        
        second_table_mapattr = {}
        for attr in select_attributes:
            second_table_mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            
            attr = remaining_attributes[0]
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+75
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        second_table_curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                                #remaining_attributes.remove(dataattr)
                            else:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                except:
                    print("error extract attributes")
                    actual_token -= len(key_sentences.split())+75
                    print("answer: ", answer) 
       
             
        if len(second_table_curdata) != 0:
            second_table_curdata['file'] = onefile
            second_table_output = second_table_output.append(second_table_curdata,ignore_index=True)
            #data = data.append(second_table_curdata,ignore_index=True)
            
            second_table_output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)



with open(second_infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

second_table_output


In [ ]:
join_attr = second_table_info.join_infor[1].join_table1_attributes
print("join_attr: ",join_attr)

second_table_output[join_attr] = second_table_output[join_attr].str.strip().astype(str)
print("second_table_output join_attr: ",second_table_output[join_attr])
join_attributes = second_table_output[join_attr].values
join_attributes = list(set(join_attributes))
print("join_attributes: ",join_attributes)

join_attributes = [str(i) for i in join_attributes]
join_attributes = ",".join(join_attributes)
join_filter_conditions = join_attr + " IN " + str(join_attributes)
print("join_attributes: ",join_attributes)
print("join_filter_conditions: ",join_filter_conditions)



In [ ]:
third_table_info, fourth_table_info = table_infos[2], table_infos[0]
third_table_path = third_table_info.file_path
fourth_table_path = fourth_table_info.file_path
#join_attr_next = second_table_info.join_infor[0].join_table1_attributes
dataC = pd.read_csv("your file path C", encoding='utf-8')
print("third_table_infor:", third_table_path)
print("fourth_table_infor:", fourth_table_path)
#print("join_attr_next: ",join_attr_next)
join_sel_third = cal_selectivity_join("join_attr","join_attr",second_table_output,dataC,is_table1_first=False)
print("join_sel_third: ",join_sel_third)

In [ ]:
third_table_filter_data = {}
third_table_output = pd.DataFrame()
all_actual_token = 0
import os
import time
third_file_list = os.listdir(third_table_path)

third_infor = result_dir + "third_infor.txt"
start_tie = time.time()
initial_where_clause = where_clause

for onefile in third_file_list:
    input_text = ""
    file_path = third_table_path + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    where_clause = third_table_info.where_clause
    where_clause = rewrite_where_clause(where_clause)
    print("where_clause: ",where_clause)

    filter_conditions = third_table_info.where_filter_conditions
    filter_cond = rewrite_where_filter_conditions(filter_conditions)
    print("filter_cond: ",filter_cond)

    data = third_table_info.data

    attributes = parse_input(input_text)

    
    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    for filter_condtion in filter_cond:
        print("filter_condtion: ",filter_condtion)
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        try:
            key_sentences = attributes[atrribute]['key_sentences']
            cost = attributes[atrribute]['cost']
        except KeyError:
            key_sentences = ""
            cost = 0
        if atrribute in attributes:
            third_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "table": third_table_info.name,
                "selectivity": selectivity,
                "cost": cost,
                's_value': calculate_s(selectivity, key_sentences),
                "key_sentences": key_sentences
            }
        else:
            third_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }
    #print("third_table_filter_data: ",third_table_filter_data)
    
    try:
            key_sentences = attributes[join_attr]['key_sentences']
            cost = attributes[join_attr]['cost']
    except KeyError:
        key_sentences = ""
        cost = 0
    if atrribute in attributes:
        third_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "table": third_table_info.name,
            "selectivity": join_sel_third,
            "cost": cost,
            's_value': calculate_s(selectivity, key_sentences),
            "key_sentences": key_sentences
        }
    else:
        third_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "selectivity": third_table_info.selectivity_join,
            's_value': 0,
            "key_sentences": []
        }


    print("third_table_filter_data: ",third_table_filter_data)
    print("where_clause: ",where_clause)
    try:
        where_clause += " AND " + join_filter_conditions
        sort_where_clause = where_clause.replace("WHERE","")
        print("sort_where_clause: ",sort_where_clause)
        sorted_filters = handle_sql(sort_where_clause, third_table_filter_data)
    except:
        print("the file is error: ",file_path)
        continue
    print("Sorted Filters:")
    for filter_cond, s_value in sorted_filters:
        print(f"{filter_cond}: S = {s_value:.4f}")
    #continue
    #break
    print("\n")

    actual_token = 0
    
    

    skipthefile = 0
    sql_copy = where_clause
    
    select_attributes = third_table_info.select_attributes
    remaining_attributes = select_attributes.copy()
    remaining_attributes.append(join_attr)
    #remaining_attributes.append(join_attr_next)
    
    remaining_attributes = list(set(remaining_attributes))
    print("remaining_attributes: ",remaining_attributes)
    third_table_curdata = {}
    third_table_sorted_filter_cond = []
    third_table_mapfilter_cond = {}
    third_table_mapfilter_erro = {}

    for filter in sorted_filters:
        third_table_sorted_filter_cond.append(filter[0])
    for filter in third_table_sorted_filter_cond:
        third_table_mapfilter_cond[filter] = 0
        third_table_mapfilter_erro[filter] = 0
    
    remaining_third_table_sorted_filter_cond = third_table_sorted_filter_cond.copy()

    while remaining_third_table_sorted_filter_cond:
        #print("remaining_third_table_sorted_filter_cond: ",remaining_third_table_sorted_filter_cond)
        filter_cond_tochange = []
        filter_cond = remaining_third_table_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        third_table_mapfilter_cond[filter_cond] += 1
        if third_table_mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remaining_third_table_sorted_filter_cond:
                remaining_third_table_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif third_table_mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remaining_third_table_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            print("answer: \n", answer)
            #attributes_cur_all = answer.split("$$")[1]
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                # if 'third_table_output NOT IN REQUIRED FORMAT' in answer and third_table_mapfilter_erro[filter_cond] >= 1:
                #     if filter_cond in remaining_third_table_sorted_filter_cond:
                #         remaining_third_table_sorted_filter_cond.remove(filter_cond)
                third_table_mapfilter_cond[filter_cond] -= 1
                third_table_mapfilter_erro[filter_cond] += 1
                tochange_filter = [filter_cond,'false']
                actual_token -= len(key_sentences.split())+100
                # filter_cond_tochange.append(tochange_filter)
                
                # for ask_filter_cond,bool_value in filter_cond_tochange:
                #         sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
                sql_copy = sql_copy.replace(filter_cond,'false')
                sql_copy2 = sql_copy
                print("sql_copy: ",sql_copy)
                bool_value_set_true = calculate_bool_value_true(sql_copy)
                bool_value_set_false = calculate_bool_value_false(sql_copy2)
                print("bool_value_set_true: ",bool_value_set_true)
                print("bool_value_set_false: ",bool_value_set_false)
                
                if bool_value_set_true == False:
                    skipthefile = 1
                    break
                
                if bool_value_set_false == True:
                    skipthefile = 0
                    break
                else:
                    skipthefile = 1
                continue

            
            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    third_table_curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            if dataattr == third_table_info.join_attributes:
                                remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remaining_third_table_sorted_filter_cond:
                            if filter_cond_cur == filter_cond:
                                #remaining_third_table_sorted_filter_cond.remove(filter_cond)
                                filter_cond_tochange.append(tochange_filter)
                        print("remaining_third_table_sorted_filter_cond: ",remaining_third_table_sorted_filter_cond)
                    else:
                        filter_cond_tochange.append([filter_cond,'false'])
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                filter_cond_tochange.append([filter_cond,'false'])
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)

        remaining_third_table_sorted_filter_cond.remove(filter_cond)
        . 
        if bool_value_set_true == False:
            skipthefile = 1
            break
        
        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")
    
    if skipthefile == 0:
        
        
        key_sentences = ""
        
        third_table_mapattr = {}
        for attr in select_attributes:
            third_table_mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            
            attr = remaining_attributes[0]
            #third_table_mapattr[attr] += 1
            # if third_table_mapattr[attr] >= 2:
            #     remaining_attributes.remove(attr)
            #     third_table_curdata[attr] = "NAN"
            #     continue
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+75
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        third_table_curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                                #remaining_attributes.remove(dataattr)
                            else:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                except:
                    print("error extract attributes")
                    actual_token -= len(key_sentences.split())+75
                    print("answer: ", answer) 
       
             
        if len(third_table_curdata) != 0:
            third_table_curdata['file'] = onefile
            third_table_output = third_table_output.append(third_table_curdata,ignore_index=True)
            #data = data.append(third_table_curdata,ignore_index=True)
            
            third_table_output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)



with open(third_infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

third_table_output


In [ ]:
print(third_table_output)

In [ ]:
dataA = pd.read_csv("your_file.csv")
#######################################################################
### join_attr you need to replace with the join attribute of yours ####
#######################################################################
join_sel_4 = cal_selectivity_join("join_attr","join_attr",second_table_output,dataA,is_table1_first=False)
join_attr_next = second_table_info.join_infor[0].join_table1_attributes
print("join_sel_4: ",join_sel_4)
print("join_attr_next: ",join_attr_next)

join_attr = second_table_info.join_infor[0].join_table1_attributes
print("join_attr: ",join_attr)

second_table_output[join_attr] = second_table_output[join_attr].str.strip().astype(str)
print("second_table_output join_attr: ",second_table_output[join_attr])
join_attributes = second_table_output[join_attr].values
join_attributes = list(set(join_attributes))
print("join_attributes: ",join_attributes)

join_attributes = [str(i) for i in join_attributes]
join_attributes = ",".join(join_attributes)
join_filter_conditions = join_attr + " IN " + str(join_attributes)
print("join_attributes: ",join_attributes)
print("join_filter_conditions: ",join_filter_conditions)


In [ ]:
fourth_table_filter_data = {}
fourth_table_output = pd.DataFrame()
all_actual_token = 0
import os
import time
fourth_file_list = os.listdir(fourth_table_path)

fourth_infor = result_dir + "fourth_infor.txt"
start_tie = time.time()
initial_where_clause = where_clause

for onefile in fourth_file_list:
    input_text = ""
    file_path = fourth_table_path + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    where_clause = fourth_table_info.where_clause
    where_clause = rewrite_where_clause(where_clause)
    print("where_clause: ",where_clause)

    filter_conditions = fourth_table_info.where_filter_conditions
    filter_cond = rewrite_where_filter_conditions(filter_conditions)
    print("filter_cond: ",filter_cond)

    data = fourth_table_info.data

    ###########################################
    
    ###########################################

    attributes = parse_input(input_text)

    
    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    for filter_condtion in filter_cond:
        print("filter_condtion: ",filter_condtion)
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        try:
            key_sentences = attributes[atrribute]['key_sentences']
            cost = attributes[atrribute]['cost']
        except KeyError:
            key_sentences = ""
            cost = 0
        if atrribute in attributes:
            fourth_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "table": fourth_table_info.name,
                "selectivity": selectivity,
                "cost": cost,
                's_value': calculate_s(selectivity, key_sentences),
                "key_sentences": key_sentences
            }
        else:
            fourth_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }
    #print("fourth_table_filter_data: ",fourth_table_filter_data)
    
    try:
            key_sentences = attributes[join_attr]['key_sentences']
            cost = attributes[join_attr]['cost']
    except KeyError:
        key_sentences = ""
        cost = 0
    if atrribute in attributes:
        fourth_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "table": fourth_table_info.name,
            "selectivity": join_sel_4,
            "cost": cost,
            's_value': calculate_s(selectivity, key_sentences),
            "key_sentences": key_sentences
        }
    else:
        fourth_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "selectivity": join_sel_4,
            's_value': 0,
            "key_sentences": []
        }


    print("fourth_table_filter_data: ",fourth_table_filter_data)
    print("where_clause: ",where_clause)
    try:
        where_clause += " AND " + join_filter_conditions
        sort_where_clause = where_clause.replace("WHERE","")
        print("sort_where_clause: ",sort_where_clause)
        sorted_filters = handle_sql(sort_where_clause, fourth_table_filter_data)
    except:
        print("the file is error: ",file_path)
        continue
    print("Sorted Filters:")
    for filter_cond, s_value in sorted_filters:
        print(f"{filter_cond}: S = {s_value:.4f}")
    #continue
    #break
    print("\n")

    actual_token = 0
    
    

    skipthefile = 0
    sql_copy = where_clause
    
    select_attributes = fourth_table_info.select_attributes
    remaining_attributes = select_attributes.copy()
    remaining_attributes.append(join_attr)
    remaining_attributes.append(join_attr_next)
    
    remaining_attributes = list(set(remaining_attributes))
    print("remaining_attributes: ",remaining_attributes)
    fourth_table_curdata = {}
    fourth_table_sorted_filter_cond = []
    fourth_table_mapfilter_cond = {}
    fourth_table_mapfilter_erro = {}

    for filter in sorted_filters:
        fourth_table_sorted_filter_cond.append(filter[0])
    for filter in fourth_table_sorted_filter_cond:
        fourth_table_mapfilter_cond[filter] = 0
        fourth_table_mapfilter_erro[filter] = 0
    
    remaining_fourth_table_sorted_filter_cond = fourth_table_sorted_filter_cond.copy()

    while remaining_fourth_table_sorted_filter_cond:
        #print("remaining_fourth_table_sorted_filter_cond: ",remaining_fourth_table_sorted_filter_cond)
        filter_cond_tochange = []
        filter_cond = remaining_fourth_table_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        fourth_table_mapfilter_cond[filter_cond] += 1
        if fourth_table_mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remaining_fourth_table_sorted_filter_cond:
                remaining_fourth_table_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif fourth_table_mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remaining_fourth_table_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            print("answer: \n", answer)
            #attributes_cur_all = answer.split("$$")[1]
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                # if 'fourth_table_output NOT IN REQUIRED FORMAT' in answer and fourth_table_mapfilter_erro[filter_cond] >= 1:
                #     if filter_cond in remaining_fourth_table_sorted_filter_cond:
                #         remaining_fourth_table_sorted_filter_cond.remove(filter_cond)
                fourth_table_mapfilter_cond[filter_cond] -= 1
                fourth_table_mapfilter_erro[filter_cond] += 1
                tochange_filter = [filter_cond,'false']
                actual_token -= len(key_sentences.split())+100
                # filter_cond_tochange.append(tochange_filter)
                
                # for ask_filter_cond,bool_value in filter_cond_tochange:
                #         sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
                sql_copy = sql_copy.replace(filter_cond,'false')
                sql_copy2 = sql_copy
                print("sql_copy: ",sql_copy)
                bool_value_set_true = calculate_bool_value_true(sql_copy)
                bool_value_set_false = calculate_bool_value_false(sql_copy2)
                print("bool_value_set_true: ",bool_value_set_true)
                print("bool_value_set_false: ",bool_value_set_false)
                
                if bool_value_set_true == False:
                    skipthefile = 1
                    break
                
                if bool_value_set_false == True:
                    skipthefile = 0
                    break
                else:
                    skipthefile = 1
                continue

            
            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    fourth_table_curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            if dataattr == fourth_table_info.join_attributes:
                                remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remaining_fourth_table_sorted_filter_cond:
                            if filter_cond_cur == filter_cond:
                                #remaining_fourth_table_sorted_filter_cond.remove(filter_cond)
                                filter_cond_tochange.append(tochange_filter)
                        print("remaining_fourth_table_sorted_filter_cond: ",remaining_fourth_table_sorted_filter_cond)
                    else:
                        filter_cond_tochange.append([filter_cond,'false'])
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                filter_cond_tochange.append([filter_cond,'false'])
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)

        remaining_fourth_table_sorted_filter_cond.remove(filter_cond)
        . 
        if bool_value_set_true == False:
            skipthefile = 1
            break
        
        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")
        #remaining_fourth_table_sorted_filter_cond.remove(filter_cond)



    
    if skipthefile == 0:
        
        
        key_sentences = ""
        
        fourth_table_mapattr = {}
        for attr in select_attributes:
            fourth_table_mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            
            attr = remaining_attributes[0]
            #fourth_table_mapattr[attr] += 1
            # if fourth_table_mapattr[attr] >= 2:
            #     remaining_attributes.remove(attr)
            #     fourth_table_curdata[attr] = "NAN"
            #     continue
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+75
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        fourth_table_curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                                #remaining_attributes.remove(dataattr)
                            else:
                                if dataattr == attr:
                                    remaining_attributes.remove(dataattr)
                except:
                    print("error extract attributes")
                    actual_token -= len(key_sentences.split())+75
                    print("answer: ", answer) 
       
             
        if len(fourth_table_curdata) != 0:
            fourth_table_curdata['file'] = onefile
            fourth_table_output = fourth_table_output.append(fourth_table_curdata,ignore_index=True)
            #data = data.append(fourth_table_curdata,ignore_index=True)
            
            fourth_table_output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)



with open(fourth_infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

fourth_table_output
